#### Shallow Clone

In [0]:
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/

In [0]:
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_c1_100
AS
SELECT * FROM
PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`

In [0]:
SELECT * from deltacatalog.deltadb.invoices_c1_100
LIMIT 5;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100;

In [0]:
DELETE FROM deltacatalog.deltadb.invoices_c1_100
where customer_id BETWEEN 15 AND 20;

In [0]:
UPDATE deltacatalog.deltadb.invoices_c1_100
SET quantity=10
where customer_id=3; 

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_c1_100
VALUES(1099,	'I178410',	'Male',	61,	'Clothing', 5,	1500.4,	'Credit Card'	,'2021-11-26', 'Metrocity',	null);

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100;

In [0]:
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_c1_100_scl SHALLOW CLONE deltacatalog.deltadb.invoices_c1_100;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100_scl;

In [0]:
UPDATE deltacatalog.deltadb.invoices_c1_100
SET quantity = 10
where customer_id=5;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100;

In [0]:
SELECT * from deltacatalog.deltadb.invoices_c1_100
where customer_id=5;

In [0]:
SELECT * from deltacatalog.deltadb.invoices_c1_100_scl
where customer_id=5;

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100_scl;

#### Deep Clone

In [0]:
DESCRIBE HISTORY deltacatalog.deltadb.invoices_c1_100;

In [0]:
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_c1_100_dcl DEEP CLONE deltacatalog.deltadb.invoices_c1_100;

#### CTAS VS DEEP CLONE

- With Deep Clone we dont need to respecify partitioning properties, the constraints and all of that.
- With CTAS table is just created using the output of a select query, all properties are lost.
- Deep Clone is a robust way to copy metadata, data and properties of a table.
- Deep Clone works in incremental manner.
- How ? Suppose I did Update and Delete on the source table after the cloning and syncing was done. So when i clone again after updates and deletes on source table it only applies update on target table and then only applies delete on target table, does not copy whole data and metadata again.
- Note : INSERT/UPDATE/DELETE/MERGE operations are synced incrementally, However SCHEMA changes or PARTITIONING, COLUMN changes will trigger a FULL DEEP CLONE.  

### Deep Clone: Incremental vs. Full Sync Behavior

| Operation Type | Examples | Sync Type |
|:---|:---|:---|
| **DML (Data)** | `INSERT`, `UPDATE`, `DELETE`, `MERGE` | **Incremental:** Only the specific changed data files are synchronized. |
| **DDL (Structure)** | `ADD COLUMN`, `RENAME COLUMN`, `DROP COLUMN` | **Full Clone:** Structural changes invalidate the file mapping, forcing a full data copy. |
| **DDL (Properties)** | Changing `PARTITIONED BY`, `TBLPROPERTIES` | **Full Clone:** Changing how the table is physically organized requires a complete rewrite. |

### Deep Clone vs. Shallow Clone Comparison

| Feature | Deep Clone | Shallow Clone |
|:---|:---|:---|
| **Data Movement** | Copies physical files. | Metadata only (pointers). |
| **Sync Mechanism** | **Incremental:** Only copies new/changed files. | **Full Refresh:** Updates the metadata pointers to the latest source state. |
| **Source Dependency** | **Independent:** Target remains if source is deleted. | **Dependent:** If source files are vacuumed, clone will break. |
| **Cost of Sync** | Proportional to the amount of data changed. | Near zero (metadata update only). |
| **Use Case** | Production migration, DR, and ETL staging. | Testing, short-term staging, and sharing snapshots. |